In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer

from xgboost import XGBClassifier

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.svm import SVC

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

data = load_breast_cancer()

X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

y = data.target

print(X.head())
print(y[:5])

In [ ]:
metabric = pd.read_csv(
    "Breast_Cancer_METABRIC.csv"
)

print(metabric.shape)
print(metabric.head())

In [ ]:
def create_effective_treatment(row):

    survival = row["Overall Survival (Months)"]

    alive = str(
        row["Patient's Vital Status"]
    ).lower()

    relapse = str(
        row["Relapse Free Status"]
    ).lower()

    if (
        survival >= 60
        and "living" in alive
        and "not" in relapse
    ):
        return 1

    return 0

metabric["Effective_Treatment"] = (
    metabric.apply(
        create_effective_treatment,
        axis=1
    )
)

In [ ]:
features = [

    "Age at Diagnosis",

    "Tumor Size",

    "Tumor Stage",

    "Neoplasm Histologic Grade",

    "Lymph nodes examined positive",

    "ER Status",

    "PR Status",

    "HER2 Status",

    "Inferred Menopausal State",

    "Chemotherapy",

    "Hormone Therapy",

    "Radio Therapy"
]

target = "Effective_Treatment"

In [ ]:
df = metabric[
    features + [target]
].copy()

X = df[features]
y = df[target]

In [ ]:
numeric_features = [

    "Age at Diagnosis",

    "Tumor Size",

    "Tumor Stage",

    "Neoplasm Histologic Grade",

    "Lymph nodes examined positive"
]

categorical_features = [

    "ER Status",

    "PR Status",

    "HER2 Status",

    "Inferred Menopausal State",

    "Chemotherapy",

    "Hormone Therapy",

    "Radio Therapy"
]

In [ ]:
preprocessor = ColumnTransformer(

    transformers=[

        (

            "num",

            Pipeline([

                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    )
                ),

                (
                    "scaler",
                    StandardScaler()
                )
            ]),

            numeric_features
        ),

        (

            "cat",

            Pipeline([

                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),

                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),

            categorical_features
        )
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y
)

In [ ]:
svm_pipeline = Pipeline([

    ("preprocessor", preprocessor),

    ("classifier",

     SVC(
         probability=True
     ))
])

In [ ]:
param_grid = {

    "classifier__C":
        [0.1, 1, 10, 100],

    "classifier__gamma":
        [1, 0.1, 0.01, 0.001],

    "classifier__kernel":
        ["rbf"]
}

In [ ]:
grid_search = GridSearchCV(
    svm_pipeline,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(
    X_train,
    y_train
)

In [ ]:
best_svm = (
    grid_search.best_estimator_
)

print(
    grid_search.best_params_
)

In [ ]:
y_pred = best_svm.predict(
    X_test
)

y_prob = (
    best_svm.predict_proba(
        X_test
    )[:,1]
)

In [ ]:
print(
    "Accuracy:",
    accuracy_score(
        y_test,
        y_pred
    )
)

print(
    classification_report(
        y_test,
        y_pred
    )
)

print(
    "ROC AUC:",
    roc_auc_score(
        y_test,
        y_prob
    )
)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
rf_pipeline = Pipeline([

    ("preprocessor", preprocessor),

    ("classifier",

     RandomForestClassifier(
         n_estimators=300,
         random_state=42
     ))
])

rf_pipeline.fit(
    X_train,
    y_train
)

rf_pred = rf_pipeline.predict(
    X_test
)

print(
    accuracy_score(
        y_test,
        rf_pred
    )
)

In [ ]:
xgb_pipeline = Pipeline([

    ("preprocessor", preprocessor),

    ("classifier",

     XGBClassifier(
         n_estimators=300,
         max_depth=5,
         learning_rate=0.05,
         eval_metric="logloss"
     ))
])

xgb_pipeline.fit(
    X_train,
    y_train
)

xgb_pred = xgb_pipeline.predict(
    X_test
)

print(
    accuracy_score(
        y_test,
        xgb_pred
    )
)

In [ ]:
external = pd.read_csv(
    "Breast_Cancer.csv"
)

external.head()

In [ ]:
external = external.rename(columns={

    "Age":
        "Age at Diagnosis",

    "Tumor Size":
        "Tumor Size",

    "Estrogen Status":
        "ER Status",

    "Progesterone Status":
        "PR Status",

    "Reginol Node Positive":
        "Lymph nodes examined positive"
})

In [ ]:
external[
    "Effective_Treatment"
] = np.where(

    external[
        "Survival Months"
    ] >= 60,

    1,

    0
)

In [ ]:
common_features = [

    "Age at Diagnosis",

    "Tumor Size",

    "ER Status",

    "PR Status",

    "Lymph nodes examined positive"
]

X_external = external[
    common_features
]

y_external = external[
    "Effective_Treatment"
]

In [ ]:
print(best_svm.feature_names_in_)
print(X_external.columns)

In [ ]:
X_external.columns

In [ ]:
external_predictions = (
    best_svm.predict(
        X_external
    )
)

print(

    accuracy_score(
        y_external,
        external_predictions
    )
)